# 7 Wonders: symulacja i trening ONNX

Ten notebook uruchamia symulację w C#, zbiera plik wynikowy z logami ruchów i trenuje hierarchiczny model policy/value na faktycznym wektorze stanu.

Przepływ:
1. `GameConsole` eksportuje dane z `MoveLog.State`, `MoveLog.ActionMask` i `MoveLog.ActionIndex`.
2. Notebook wczytuje najnowszy plik `training_*.json`.
3. Model PyTorch uczy się i eksportuje `policy_network.onnx`.

In [ ]:
from pathlib import Path
import subprocess
import sys

repo_root = Path(r"c:/Users/kubeu/Kuba-dokumenty/Magisterka/7 Wonders")
game_console = repo_root / "GameConsole" / "GameConsole.csproj"
results_dir = repo_root / "GameConsole" / "Results"
encoding_dir = repo_root / "GameAI" / "Encoding"

sys.path.append(str(encoding_dir))

from game_training_pipeline import ActionSpace, GameDataset, HierarchicalPolicyNetwork, train_epoch, evaluate

print("Repo root:", repo_root)
print("State vector size:", ActionSpace.STATE_VECTOR_SIZE)
print("Primary action size:", ActionSpace.TOTAL_PRIMARY_ACTIONS)

In [ ]:
def run_simulation(seed: int = 12345, games: int = 20, agent1: str = "heuristic-personal", agent2: str = "mcts"):
    results_dir.mkdir(parents=True, exist_ok=True)
    command = [
        "dotnet", "run",
        "--project", str(game_console),
        "--",
        "export-data",
        "--seed", str(seed),
        "--games", str(games),
        "--agent1", agent1,
        "--agent2", agent2,
    ]
    subprocess.run(command, cwd=repo_root, check=True)

run_simulation(seed=12345, games=20)
print("Simulation finished.")

In [ ]:
from torch.utils.data import DataLoader, random_split
import torch

training_files = sorted(results_dir.glob("training_*.json"))
if not training_files:
    raise FileNotFoundError(f"No training_*.json files found in {results_dir}")

latest_file = max(training_files, key=lambda p: p.stat().st_mtime)
print("Using dataset:", latest_file)

dataset = GameDataset(str(latest_file), normalize=True, validate_shapes=True)
train_size = max(1, int(len(dataset) * 0.9))
val_size = max(1, len(dataset) - train_size)
if train_size + val_size > len(dataset):
    val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False) if len(val_dataset) > 0 else None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HierarchicalPolicyNetwork(state_dim=ActionSpace.STATE_VECTOR_SIZE, hidden_dim=256, dropout=0.1).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(5):
    train_stats = train_epoch(model, train_loader, optimizer, device)
    if val_loader is not None:
        val_stats = evaluate(model, val_loader, device)
        print(f"epoch={epoch + 1} train={train_stats['total_loss']:.4f} val={val_stats['total_loss']:.4f}")
    else:
        print(f"epoch={epoch + 1} train={train_stats['total_loss']:.4f}")

In [ ]:
onnx_path = encoding_dir / "policy_network.onnx"
model.onnx_export(str(onnx_path), validate=True)
print("Exported:", onnx_path)

batch = next(iter(train_loader))
state = batch['state'].to(device)
action_mask = batch['action_mask'].to(device)
outputs = model(state, action_mask=action_mask)
print("policy shape:", outputs['policy_masked_logits'].shape)
print("value shape:", outputs['value'].shape)